# RSNA Knee — frozen MAE inference

Inference only: the 30-epoch MAE encoder, mean pooling over all slices, and the average of five saved logistic classifiers. No training, reports, test labels, downloads, or network calls.

**Kaggle setup:** upload the contents of `submission/mae_assets` as a Kaggle dataset (or upload `submission/mae_assets.zip` and ensure it is extracted). Also upload `submission/dicom_codecs.zip` as a second dataset named **RSNA DICOM Codecs** and ensure its wheels are extracted. Attach the two datasets and the competition to this notebook. Restart the session before running this updated notebook. Select a GPU, disable Internet, and run all cells. Output: `/kaggle/working/submission.csv`. Set the two directory overrides below if automatic discovery is ambiguous. Required installed libraries: PyTorch, NumPy, pandas, Pillow, OpenCV, pydicom and appropriate DICOM pixel decoders. The first cell installs the attached GDCM wheel offline with `--no-index --no-deps`, before importing pydicom. It does not download packages or change NumPy.

**Local run:** run from the project root; the notebook uses the three example-test studies' existing JPEGs and writes `submission/local_example/submission.csv`. That checks inference and format, not leaderboard performance. Local DICOM checks are in `scripts/verification/verify_mae_submission.py`.

The best observed local OOF macro AUC is **0.5741 on 58 studies**, not a Kaggle score. This notebook ensembles fold probabilities; its hidden-test performance is unknown. The JPEG conversion was inferred from 25 paired examples, with a very small nonzero pixel difference; generalization to all scanners and Kaggle decoder versions still needs a remote run.

In [ ]:
# Offline codec setup: run BEFORE importing pydicom so its plugins see GDCM.
from pathlib import Path
import hashlib
import importlib
import json
import os
import subprocess
import sys


def install_offline_gdcm(input_root, target_dir, python_tag=None):
    python_tag = python_tag or f"cp{sys.version_info.major}{sys.version_info.minor}"
    candidates = []
    for directory, subdirs, files in os.walk(input_root):
        path = Path(directory)
        depth = len(path.relative_to(input_root).parts)
        subdirs[:] = [s for s in subdirs if s not in {
            "competitions", "rsna-knee-abnormality-detection", "train_series", "test_series"}]
        if depth >= 5:
            subdirs[:] = []
        if "codecs_manifest.json" in files:
            metadata = json.loads((path / "codecs_manifest.json").read_text())
            if metadata.get("package") != "python-gdcm" or metadata.get("version") != "3.0.26":
                continue
            for name, expected in metadata["files"].items():
                if Path(name).name != name:
                    raise ValueError("Invalid codec filename")
                if f"-{python_tag}-{python_tag}-" in name and "x86_64.whl" in name:
                    candidates.append((path / name, expected))
    if not candidates:
        raise RuntimeError(
            f"No offline GDCM wheel for {python_tag}. Attach the extracted dicom_codecs.zip "
            "as a Kaggle input dataset, then restart and run all cells. "
            "The provided bundle supports Linux Python 3.10 through 3.13.")
    wheel, expected = sorted(candidates, key=lambda item: str(item[0]))[0]
    if hashlib.sha256(wheel.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Codec wheel checksum mismatch: {wheel.name}")
    target_dir = Path(target_dir)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
        "--disable-pip-version-check", "--upgrade", "--target", str(target_dir), str(wheel),
    ], check=True)
    sys.path.insert(0, str(target_dir))
    importlib.invalidate_caches()
    return wheel.name


if Path("/kaggle/input").is_dir():
    selected_wheel = install_offline_gdcm(Path("/kaggle/input"), Path("/kaggle/working/_dicom_codecs"))
    import gdcm
    print(f"Offline decoder ready: GDCM {gdcm.Version.GetVersion()} ({selected_wheel})")

# End offline codec setup.

from pathlib import Path
import hashlib
import io
import json
import os
import time
import warnings

import cv2
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
import torch
from torch import nn

# Set these to explicit directories only if discovery reports ambiguity.
COMPETITION_DIR_OVERRIDE = None
ASSET_DIR_OVERRIDE = None
INPUT_FORMAT_OVERRIDE = None  # None, "dicom", or "jpeg"; Kaggle requires dicom.
BATCH_SIZE = 32
DEVICE_OVERRIDE = None  # None selects CUDA, then MPS, then CPU.

ON_KAGGLE = Path("/kaggle/input").is_dir()
# Resolve the repository from the root or this notebook's folder.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "baseline_candidate").is_dir() and (p / "notebooks").is_dir()), Path.cwd())
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else PROJECT_ROOT / "submission/local_example"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
# Remove stale output before beginning; a failed run must not expose old predictions.
SUBMISSION_PATH.unlink(missing_ok=True)

if COMPETITION_DIR_OVERRIDE is not None:
    COMPETITION_DIR = Path(COMPETITION_DIR_OVERRIDE)
else:
    candidates = ([Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                   Path("/kaggle/input/rsna-knee-abnormality-detection")]
                  if ON_KAGGLE else [PROJECT_ROOT / "data"])
    found = [p for p in candidates if (p / "test.csv").is_file()]
    if len(found) != 1:
        raise RuntimeError(f"Set COMPETITION_DIR_OVERRIDE; found {found}")
    COMPETITION_DIR = found[0]

if ASSET_DIR_OVERRIDE is not None:
    ASSET_DIR = Path(ASSET_DIR_OVERRIDE)
elif not ON_KAGGLE:
    ASSET_DIR = PROJECT_ROOT / "submission/mae_assets"
else:
    found = []
    # Search dataset mount layouts without descending into competition DICOM trees.
    for directory, subdirs, files in os.walk("/kaggle/input"):
        path = Path(directory)
        depth = len(path.relative_to("/kaggle/input").parts)
        subdirs[:] = [s for s in subdirs if s not in {
            "competitions", "rsna-knee-abnormality-detection", "train_series", "test_series"}]
        if depth >= 4:
            subdirs[:] = []
        if {"manifest.json", "encoder.pt", "config.json"}.issubset(files):
            found.append(path)
    if len(found) != 1:
        raise RuntimeError(f"Set ASSET_DIR_OVERRIDE to the extracted MAE assets; found {found}")
    ASSET_DIR = found[0]

INPUT_FORMAT = INPUT_FORMAT_OVERRIDE or ("dicom" if ON_KAGGLE else "jpeg")
if INPUT_FORMAT not in {"dicom", "jpeg"} or (ON_KAGGLE and INPUT_FORMAT != "dicom"):
    raise ValueError("Kaggle hidden inference requires INPUT_FORMAT='dicom'")
DEVICE = torch.device(DEVICE_OVERRIDE or (
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"))
torch.set_num_threads(min(4, os.cpu_count() or 1))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
print({"competition": str(COMPETITION_DIR), "assets": str(ASSET_DIR),
       "format": INPUT_FORMAT, "device": str(DEVICE)})

## Verify and load the small inference bundle

The bundle contains only encoder tensors, classifier tensors, and configuration/provenance. It omits the decoder, optimizer, reports, images, and study identifiers. Hash validation detects incomplete or changed files. Only tensor-safe checkpoint loading is used.

In [ ]:
manifest = json.loads((ASSET_DIR / "manifest.json").read_text())
if manifest["schema_version"] != 1 or manifest["representation"] != "mae_mean" or manifest["folds"] != 5:
    raise ValueError("Unsupported inference bundle")
expected_files = {"config.json", "encoder.pt"} | {f"probe_fold{i}.pt" for i in range(5)}
if set(manifest["files"]) != expected_files:
    raise ValueError("Bundle file manifest is incomplete or unexpected")
for name, expected in manifest["files"].items():
    if hashlib.sha256((ASSET_DIR / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Checksum mismatch: {name}")
CFG = json.loads((ASSET_DIR / "config.json").read_text())
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
assert manifest["labels"] == LABELS
assert CFG["image_size"] == 224 and CFG["encoder_dim"] == 192

class Block(nn.Module):
    def __init__(self, dim, heads, ratio):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, ratio*dim), nn.GELU(), nn.Linear(ratio*dim, dim))

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

class MAEEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.patch_size = cfg["patch_size"]
        dim = cfg["encoder_dim"]
        self.patch_embed = nn.Linear(self.patch_size**2, dim)
        self.register_buffer("encoder_pos", torch.zeros(1, (cfg["image_size"] // self.patch_size)**2, dim))
        self.encoder = nn.Sequential(*[
            Block(dim, cfg["encoder_heads"], cfg["mlp_ratio"]) for _ in range(cfg["encoder_depth"])])
        self.encoder_norm = nn.LayerNorm(dim)

    def forward(self, x):
        b, c, h, w = x.shape
        p = self.patch_size
        assert c == 1 and h == w == CFG["image_size"]
        g = h // p
        patches = x.reshape(b, c, g, p, g, p).permute(0, 2, 4, 3, 5, 1).reshape(b, g*g, p*p)
        tokens = self.patch_embed(patches) + self.encoder_pos
        return self.encoder_norm(self.encoder(tokens)).mean(dim=1)

encoder = MAEEncoder(CFG)
encoder.load_state_dict(torch.load(ASSET_DIR / "encoder.pt", map_location="cpu", weights_only=True), strict=True)
encoder = encoder.to(DEVICE).eval().requires_grad_(False)
probes = []
for fold in range(5):
    probe = torch.load(ASSET_DIR / f"probe_fold{fold}.pt", map_location="cpu", weights_only=True)
    assert probe["labels"] == LABELS and probe["representation"] == "mae_mean"
    for key, shape in {"weights": (12, 192), "biases": (12,), "mean": (192,), "scale": (192,)}.items():
        value = probe[key]
        assert tuple(value.shape) == shape and torch.isfinite(value).all()
        probe[key] = value.double()
    assert (probe["scale"] > 0).all()
    probes.append(probe)
print(f"Loaded encoder and {len(probes)} classifiers; no training required.")

## Match the training pixels

For each grayscale frame: float32 min–max scaling → uint8 → OpenCV `INTER_AREA` resize → JPEG quality 95 round trip → Pillow decode → divide by 255. No new physical crop, windowing, inversion, or orientation transform is introduced because those were not established in the training JPEGs. Slice ordering has no anatomical role here: this model mean-pools every image independently.

Unreadable slices are recorded in `decode_errors.csv` and skipped. A missing study directory is a hard error. A study with files but no decodable frames receives neutral 0.5 predictions, prominently logged. Missing decoder dependencies are a hard error, not a reason to silently replace predictions. No limit on slices is imposed.

In [ ]:
# Fail early if the environment cannot decode the common competition syntaxes.
# The local JPEG check does not require any DICOM codec.
if INPUT_FORMAT == "dicom":
    from pydicom.pixels import get_decoder
    from pydicom.uid import JPEGLosslessSV1, JPEG2000Lossless
    missing = []
    for syntax in [JPEGLosslessSV1, JPEG2000Lossless]:
        decoder = get_decoder(syntax)
        if not decoder.is_available:
            missing.append(f"{syntax.name}: {decoder.missing_dependencies}")
    if missing:
        raise ImportError("DICOM codecs unavailable. Attach the provided dicom_codecs.zip dataset, "
                          "restart the session, and run the updated first cell; or use an environment "
                          "with compatible decoders. " + "; ".join(missing))

def pixel_array_to_training_tensor(frame):
    frame = np.asarray(frame, dtype=np.float32)
    if frame.ndim != 2 or not frame.size or not np.isfinite(frame).all():
        raise ValueError("Expected a finite, nonempty grayscale frame")
    lo, hi = float(frame.min()), float(frame.max())
    normalized = (frame - lo) / (hi - lo) if hi > lo else np.zeros_like(frame)
    pixels = (normalized * 255).astype(np.uint8)
    pixels = cv2.resize(pixels, (224, 224), interpolation=cv2.INTER_AREA)
    ok, encoded = cv2.imencode(".jpg", pixels, [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        raise RuntimeError("JPEG conversion failed")
    with Image.open(io.BytesIO(encoded.tobytes())) as image:
        pixels = np.array(image, dtype=np.float32, copy=True) / 255.0
    return torch.from_numpy(pixels).unsqueeze(0)


def read_frames(path, input_format):
    if input_format == "jpeg":
        with Image.open(path) as image:
            if image.mode != "L" or image.size != (224, 224):
                raise ValueError(f"Training JPEG contract violated: {image.mode}, {image.size}")
            array = np.array(image, dtype=np.float32, copy=True) / 255.0
        yield torch.from_numpy(array).unsqueeze(0)
    else:
        dataset = pydicom.dcmread(path)
        if int(getattr(dataset, "SamplesPerPixel", 1)) != 1:
            raise ValueError("Color DICOM is outside the grayscale training contract")
        try:
            pixels = dataset.pixel_array
        except RuntimeError as exc:
            message = str(exc).lower()
            if any(term in message for term in ["missing dependencies", "no plugins", "all plugins", "requires", "unavailable"]):
                raise ImportError(f"Required pixel decoder unavailable for {dataset.file_meta.TransferSyntaxUID}: {exc}") from exc
            raise
        if pixels.ndim == 2:
            yield pixel_array_to_training_tensor(pixels)
        elif pixels.ndim == 3 and pixels.shape[0] == int(getattr(dataset, "NumberOfFrames", 1)):
            for frame in pixels:
                yield pixel_array_to_training_tensor(frame)
        else:
            raise ValueError(f"Unsupported DICOM pixel shape: {pixels.shape}")


def study_feature(paths, input_format):
    accumulator = torch.zeros(CFG["encoder_dim"], dtype=torch.float64)
    count, batch, errors = 0, [], []

    def consume(images):
        with torch.inference_mode():
            features = encoder(torch.stack(images).to(DEVICE)).cpu()
        if not torch.isfinite(features).all():
            raise RuntimeError("Nonfinite encoder features")
        return features.double().sum(dim=0)

    for path in paths:
        # Only decode exceptions are recoverable; model/OOM errors must stop the run.
        iterator = iter(read_frames(path, input_format))
        while True:
            try:
                frame = next(iterator)
            except StopIteration:
                break
            except ImportError:
                raise
            except Exception as exc:
                errors.append({"file": str(path), "error": f"{type(exc).__name__}: {exc}"})
                break
            batch.append(frame)
            if len(batch) == BATCH_SIZE:
                accumulator += consume(batch)
                count += len(batch)
                batch = []
    if batch:
        accumulator += consume(batch)
        count += len(batch)
    # Match the baseline's float64 accumulation followed by float32 feature storage.
    feature = (accumulator / count).float().numpy() if count else None
    return feature, count, errors


def fold_probabilities(feature):
    x = torch.as_tensor(feature, dtype=torch.float64).reshape(1, -1)
    predictions = []
    for probe in probes:
        z = (x - probe["mean"]) / probe["scale"]
        predictions.append((z @ probe["weights"].T + probe["biases"]).sigmoid().numpy()[0])
    return np.stack(predictions)


def predict_feature(feature):
    return np.full(len(LABELS), 0.5) if feature is None else fold_probabilities(feature).mean(axis=0)

## Predict every requested test study and write the submission

Test IDs are read at runtime; example IDs and sample row counts are never hardcoded. All available series and frames are included. Accumulation is streamed in batches. The output is written atomically only after ID, column, shape, finiteness and probability checks pass. The report includes timing but does not claim a full hidden-test runtime guarantee.

In [ ]:
test = pd.read_csv(COMPETITION_DIR / "test.csv", dtype={"StudyInstanceUID": str})
if "StudyInstanceUID" not in test or test.empty or test.StudyInstanceUID.isna().any() or test.StudyInstanceUID.duplicated().any():
    raise ValueError("Invalid test study IDs")
ids = test.StudyInstanceUID.tolist()
if any(Path(uid).name != uid or uid in {".", ".."} for uid in ids):
    raise ValueError("Invalid study path component")
rows, study_log, decode_errors = [], [], []
started = time.perf_counter()
for index, study_id in enumerate(ids):
    study_dir = COMPETITION_DIR / "test_series" / study_id
    suffix = "*.dcm" if INPUT_FORMAT == "dicom" else "*.jpeg"
    paths = sorted(study_dir.glob(f"*/{suffix}"))
    if not paths:
        raise FileNotFoundError(f"No {INPUT_FORMAT} files found in {study_dir}")
    study_started = time.perf_counter()
    feature, count, errors = study_feature(paths, INPUT_FORMAT)
    if errors:
        warnings.warn(f"{study_id}: {len(errors)} files failed decoding")
    if feature is None:
        warnings.warn(f"{study_id}: ALL files failed; using neutral 0.5 predictions")
    probabilities = predict_feature(feature)
    rows.append([study_id, *probabilities.tolist()])
    study_log.append({"StudyInstanceUID": study_id, "files": len(paths), "frames": count,
                      "decode_failures": len(errors), "neutral_fallback": feature is None,
                      "seconds": time.perf_counter() - study_started})
    decode_errors.extend({"StudyInstanceUID": study_id, **error} for error in errors)
    if index < 3 or (index + 1) % 25 == 0 or index + 1 == len(ids):
        print(f"{index+1}/{len(ids)} studies; {count} frames in last study; elapsed {time.perf_counter()-started:.1f}s")

submission = pd.DataFrame(rows, columns=["StudyInstanceUID", *LABELS])
assert submission.StudyInstanceUID.tolist() == ids
values = submission[LABELS].to_numpy(dtype=np.float64)
assert values.shape == (len(ids), 12) and np.isfinite(values).all()
assert ((values >= 0) & (values <= 1)).all()
sample_path = COMPETITION_DIR / "sample_submission.csv"
if sample_path.exists():
    assert pd.read_csv(sample_path, nrows=0).columns.tolist() == submission.columns.tolist()
pd.DataFrame(study_log).to_csv(OUTPUT_DIR / "study_diagnostics.csv", index=False)
pd.DataFrame(decode_errors, columns=["StudyInstanceUID", "file", "error"]).to_csv(OUTPUT_DIR / "decode_errors.csv", index=False)
report = {"studies": len(ids), "frames": sum(row["frames"] for row in study_log),
          "decode_failures": len(decode_errors),
          "neutral_fallback_studies": sum(row["neutral_fallback"] for row in study_log),
          "inference_seconds": time.perf_counter() - started, "device": str(DEVICE),
          "input_format": INPUT_FORMAT, "source_run": manifest["source_run"],
          "versions": {"torch": torch.__version__, "pydicom": pydicom.__version__, "opencv": cv2.__version__},
          "leaderboard_score": None}
(OUTPUT_DIR / "inference_report.json").write_text(json.dumps(report, indent=2) + "\n")
temporary = SUBMISSION_PATH.with_suffix(".csv.tmp")
submission.to_csv(temporary, index=False)
check = pd.read_csv(temporary, dtype={"StudyInstanceUID": str})
assert check.StudyInstanceUID.tolist() == ids and check.columns.tolist() == submission.columns.tolist()
np.testing.assert_allclose(check[LABELS], values, rtol=1e-12, atol=1e-15)
temporary.replace(SUBMISSION_PATH)
print(f"Saved {SUBMISSION_PATH}; no Kaggle score is available until submission.")
display(submission.head())